# SignalGate - GPU Colab dalam SATU sel

1. **Runtime > Change runtime type > T4 GPU** (atau GPU lain), lalu **Runtime > Run all**. Hanya ada satu sel kode.
2. Sel itu memeriksa GPU, memasang dan menyalakan Ollama (hanya di `127.0.0.1`), mengunduh model, menyalakan **gateway bertoken** di port 11435, menguji gateway, membuka tunnel Cloudflare ke gateway, lalu mencetak **URL layanan** dan **token gateway**.
3. Di laptop: dashboard SignalGate, tab **Runtime & GPU** -> **GPU Colab** -> tempel URL (bukan link halaman notebook ini) dan token -> **Cek kesiapan** -> **Aktifkan untuk run berikutnya**.

Aman dijalankan ulang di sesi yang sama: instalasi dan model yang sudah ada dilewati; gateway dan tunnel dimulai ulang (URL baru, jadi tempel ulang di dashboard). Jangan bagikan output sel (berisi token). Token tetap bisa disimpan di Colab Secrets bernama `SIGNALGATE_GATEWAY_TOKEN` (minimal 24 karakter); tanpa itu token acak baru dibuat tiap kali sel dijalankan.

Gateway hanya meneruskan endpoint yang dipakai SignalGate (tags, version, ps, chat, generate, show); pull/delete/create/copy/push ditolak dari luar. Port 8080 sengaja tidak dipakai: di Colab port itu milik Jupyter Server.

In [ ]:
# SignalGate: satu sel untuk menyiapkan GPU Colab. Aman dijalankan ulang di sesi yang sama.
import json, os, re, secrets, shutil, socket, subprocess, time, urllib.error, urllib.parse, urllib.request

MODELS = ["qwen2.5:14b", "glm4:9b", "gemma3:12b"]  # analis + dua pembanding (profil workstation)
OLLAMA = "http://127.0.0.1:11434"
PORT = 11435  # gateway; BUKAN 8080 -- port itu milik Jupyter Server Colab
LOCAL = f"http://127.0.0.1:{PORT}"
EXPECTED = {"tanpa token -> 401": 401, "dengan token -> 200": 200, "pull diblokir -> 403": 403,
            "info GPU -> 200": 200}

# Salinan persis colab/ollama_gateway.py dari repo (test backend memastikan isinya sama).
GATEWAY_SOURCE = r'''"""Gateway autentikasi di depan Ollama untuk tunnel Colab SignalGate.

Tunnel Cloudflare meneruskan ke gateway ini, BUKAN ke Ollama langsung (Ollama hanya mendengar di
127.0.0.1). Setiap request wajib membawa `Authorization: Bearer <token>`; yang diteruskan hanya endpoint
yang dipakai SignalGate. Pengelolaan model (pull, delete, create, copy, push, blobs) dan endpoint lain
ditolak. Header Authorization tidak pernah dicatat ke log.

Menjalankan:  SIGNALGATE_GATEWAY_TOKEN=<token> python ollama_gateway.py --port 11435
Port 8080 sengaja dihindari: di runtime Colab port itu sudah dipakai layanan internal Colab.
File ini disalin apa adanya ke notebook colab/signalgate_gpu_setup.ipynb; test backend memastikan isinya sama.
"""
from __future__ import annotations

import argparse
import hmac
import http.client
import json
import os
import subprocess
import sys
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

ALLOWED = {("GET", "/api/version"), ("GET", "/api/tags"), ("GET", "/api/ps"), ("POST", "/api/chat"),
           ("POST", "/api/generate"), ("POST", "/api/show")}
MAX_BODY = 8 * 1024 * 1024
CHUNK = 8192
MIN_TOKEN_CHARS = 24
DEFAULT_PORT = 11435
# Penanda di setiap jawaban gateway. Header `Server` tidak bisa dipakai: Cloudflare menggantinya dengan
# "cloudflare". Dengan penanda ini backend bisa membedakan gateway SignalGate dari layanan lain di URL itu.
MARKER_HEADER = "X-SignalGate-Gateway"


def gpu_info() -> list[dict]:
    """GPU yang terlihat nvidia-smi di runtime ini; kosong bila runtime tanpa GPU."""
    try:
        output = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                                capture_output=True, text=True, timeout=10, check=True).stdout
    except (OSError, subprocess.SubprocessError):
        return []
    gpus = []
    for line in output.strip().splitlines():
        name, _, memory = line.rpartition(",")
        try:
            gpus.append({"name": name.strip(), "memory_total_mb": int(float(memory.strip()))})
        except ValueError:
            gpus.append({"name": line.strip(), "memory_total_mb": None})
    return gpus


def make_handler(token: str, upstream: tuple[str, int] = ("127.0.0.1", 11434), gpus=gpu_info):
    expected = f"Bearer {token}".encode()

    class Handler(BaseHTTPRequestHandler):
        # HTTP/1.0: respons stream (NDJSON dari /api/chat) diakhiri dengan menutup koneksi.
        protocol_version = "HTTP/1.0"
        server_version = "SignalGateGateway/1"
        sys_version = ""

        def log_message(self, format, *args):  # noqa: A002 - tanda tangan dari BaseHTTPRequestHandler
            # Hanya metode, path, dan status; tidak pernah header (token) atau badan request.
            sys.stderr.write(f"gateway {self.command} {self.path.split('?', 1)[0]} {args[1] if len(args) > 1 else ''}\n")

        def end_headers(self) -> None:
            self.send_header(MARKER_HEADER, "1")
            super().end_headers()

        def _json(self, status: int, payload: dict, headers: tuple = ()) -> None:
            body = json.dumps(payload).encode()
            self.send_response(status)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            for key, value in headers:
                self.send_header(key, value)
            self.end_headers()
            self.wfile.write(body)

        def _authorized(self) -> bool:
            return hmac.compare_digest((self.headers.get("Authorization") or "").encode(), expected)

        def _handle(self, method: str) -> None:
            path = self.path.split("?", 1)[0]
            if not self._authorized():
                self._json(401, {"error": "token gateway salah atau tidak ada"}, (("WWW-Authenticate", "Bearer"),))
                return
            if method == "GET" and path == "/gateway/info":
                self._json(200, {"gpus": gpus(), "allowed": sorted(f"{m} {p}" for m, p in ALLOWED)})
                return
            if (method, path) not in ALLOWED:
                self._json(403, {"error": f"{method} {path} tidak diizinkan lewat gateway SignalGate"})
                return
            try:
                length = int(self.headers.get("Content-Length") or 0)
            except ValueError:
                length = -1
            if length < 0 or length > MAX_BODY:
                self._json(413, {"error": "badan request tidak sah atau terlalu besar"})
                return
            body = self.rfile.read(length) if length else None
            connection = http.client.HTTPConnection(*upstream, timeout=900)
            started = False
            try:
                # Ollama memeriksa Host; permintaan diteruskan seolah dari localhost.
                connection.request(method, path, body=body,
                                   headers={"Content-Type": self.headers.get("Content-Type") or "application/json",
                                            "Host": f"localhost:{upstream[1]}"})
                response = connection.getresponse()
                self.send_response(response.status)
                self.send_header("Content-Type", response.getheader("Content-Type") or "application/json")
                self.end_headers()
                started = True
                while True:
                    chunk = response.read1(CHUNK)
                    if not chunk:
                        break
                    self.wfile.write(chunk)
                    self.wfile.flush()
            except (OSError, http.client.HTTPException):
                if not started:
                    self._json(502, {"error": "Ollama di runtime ini tidak menjawab"})
            finally:
                connection.close()

        def do_GET(self) -> None:  # noqa: N802
            self._handle("GET")

        def do_POST(self) -> None:  # noqa: N802
            self._handle("POST")

        def do_DELETE(self) -> None:  # noqa: N802
            self._handle("DELETE")

        def do_PUT(self) -> None:  # noqa: N802
            self._handle("PUT")

        def do_HEAD(self) -> None:  # noqa: N802
            self._handle("HEAD")

    return Handler


def make_server(host: str, port: int, token: str, upstream: tuple[str, int] = ("127.0.0.1", 11434),
                gpus=gpu_info) -> ThreadingHTTPServer:
    if len(token) < MIN_TOKEN_CHARS:
        raise ValueError(f"Token gateway minimal {MIN_TOKEN_CHARS} karakter.")
    server = ThreadingHTTPServer((host, port), make_handler(token, upstream, gpus))
    server.daemon_threads = True
    return server


def main() -> None:
    parser = argparse.ArgumentParser(description="Gateway autentikasi Ollama untuk SignalGate")
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=DEFAULT_PORT)
    parser.add_argument("--upstream-port", type=int, default=11434)
    args = parser.parse_args()
    token = os.environ.get("SIGNALGATE_GATEWAY_TOKEN", "")
    try:
        server = make_server(args.host, args.port, token, ("127.0.0.1", args.upstream_port))
    except ValueError as error:
        sys.exit(str(error))
    sys.stderr.write(f"gateway siap di {args.host}:{args.port} -> Ollama 127.0.0.1:{args.upstream_port}\n")
    server.serve_forever()


if __name__ == "__main__":
    main()
'''


def step(title):
    print(f"\n== {title}", flush=True)


def tail(path, lines=20):
    try:
        with open(path, encoding="utf-8", errors="replace") as handle:
            return "".join(handle.readlines()[-lines:])
    except OSError as error:
        return f"({path} tidak terbaca: {error})"


def sh(script):
    # Output proses anak tidak otomatis tampil di notebook, jadi ditangkap lalu dicetak.
    result = subprocess.run(["bash", "-c", script], capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if output:
        print(output[-3000:])
    if result.returncode != 0:
        raise SystemExit(f"Perintah gagal (kode {result.returncode}).")


def ollama(path, timeout=10):
    with urllib.request.urlopen(OLLAMA + path, timeout=timeout) as response:
        return json.loads(response.read() or b"{}")


def ollama_up():
    try:
        ollama("/api/version", timeout=3)
        return True
    except Exception:
        return False


def status(base, path, with_token, method="GET"):
    # Kode HTTP bila yang menjawab gateway SignalGate; layanan lain ditandai; tanpa jawaban -> penjelasan error.
    request = urllib.request.Request(base + path, method=method, data=b"{}" if method == "POST" else None)
    if with_token:
        request.add_header("Authorization", "Bearer " + token)
    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            code, headers = response.status, response.headers
    except urllib.error.HTTPError as error:
        code, headers = error.code, error.headers
    except Exception as error:
        return f"tanpa jawaban HTTP ({type(error).__name__}: {getattr(error, 'reason', error)})"
    if (headers or {}).get("X-SignalGate-Gateway") != "1":
        return f"{code} dari layanan LAIN, bukan gateway SignalGate"
    return code


def checks(base):
    return {"tanpa token -> 401": status(base, "/api/tags", False),
            "dengan token -> 200": status(base, "/api/tags", True),
            "pull diblokir -> 403": status(base, "/api/pull", True, "POST"),
            "info GPU -> 200": status(base, "/gateway/info", True)}


# 1) GPU
step("GPU runtime")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if gpu is None or gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit("Runtime ini TANPA GPU. Runtime > Change runtime type > T4 GPU, lalu jalankan ulang sel ini.")
print(gpu.stdout.strip())

# 2) Ollama, hanya di 127.0.0.1 (akses dari luar lewat gateway bertoken)
step("Ollama")
if not shutil.which("ollama"):
    print("Memasang Ollama ...", flush=True)
    sh('''set -e
curl -sSL https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst -o /content/ollama.tar.zst
if ! tar --zstd -C /usr -xf /content/ollama.tar.zst 2>/content/tar_error.log; then
  apt-get install -y -qq zstd > /dev/null
  zstd -d -f /content/ollama.tar.zst -o /content/ollama.tar
  tar -C /usr -xf /content/ollama.tar
fi
ollama --version''')
if not ollama_up():
    ollama_log = open("/content/ollama.log", "a")
    subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
                     stdout=ollama_log, stderr=ollama_log, start_new_session=True)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
    else:
        print(tail("/content/ollama.log"))
        raise SystemExit("Ollama gagal start dalam 60 detik.")
print("Ollama", ollama("/api/version").get("version"), "berjalan di 127.0.0.1:11434")

# 3) Model (yang sudah ada dilewati)
step("Model")
installed = {item["name"] for item in ollama("/api/tags").get("models", [])}
for model in MODELS:
    if model in installed or f"{model}:latest" in installed:
        print(f"{model}: sudah ada")
        continue
    print(f"{model}: mengunduh ...", flush=True)
    request = urllib.request.Request(OLLAMA + "/api/pull", data=json.dumps({"model": model, "stream": True}).encode(),
                                     headers={"Content-Type": "application/json"})
    shown, layer = -10, None
    with urllib.request.urlopen(request, timeout=3600) as response:
        for raw in response:
            event = json.loads(raw or b"{}")
            if event.get("error"):
                raise SystemExit(f"Gagal mengunduh {model}: {event['error']}")
            if event.get("digest") != layer:
                shown, layer = -10, event.get("digest")
            total, done = event.get("total"), event.get("completed")
            if total and done:
                percent = int(done * 100 / total)
                if percent >= shown + 10:
                    print(f"  {percent}%", flush=True)
                    shown = percent
    print(f"{model}: selesai")

# 4) Gateway bertoken (menggantikan gateway/tunnel lama, termasuk versi lama di port 8080)
step(f"Gateway bertoken di 127.0.0.1:{PORT}")
with open("/content/ollama_gateway.py", "w", encoding="utf-8") as handle:
    handle.write(GATEWAY_SOURCE)
token = ""
try:
    from google.colab import userdata  # Colab Secrets (opsional)
    token = userdata.get("SIGNALGATE_GATEWAY_TOKEN") or ""
except Exception:
    token = ""
if len(token) < 24:
    token = secrets.token_urlsafe(32)
subprocess.run(["pkill", "-f", "ollama_gateway.py"], check=False)
subprocess.run(["pkill", "-x", "cloudflared"], check=False)  # -x: cocokkan NAMA proses, sel ini tidak ikut mati
time.sleep(1)
gateway_log = open("/content/gateway.log", "w")
subprocess.Popen(["python3", "/content/ollama_gateway.py", "--host", "127.0.0.1", "--port", str(PORT)],
                 env={**os.environ, "SIGNALGATE_GATEWAY_TOKEN": token}, stdout=gateway_log, stderr=gateway_log,
                 start_new_session=True)
local = {}
for _ in range(20):
    local = checks(LOCAL)
    if local == EXPECTED:
        break
    time.sleep(1)
for name, code in local.items():
    print(f"  {name}: {code}")
if local != EXPECTED:
    print("\n--- /content/gateway.log ---\n" + tail("/content/gateway.log"))
    print("--- /content/ollama.log ---\n" + tail("/content/ollama.log"))
    sh(f"ss -ltnp | grep ':{PORT} ' || echo '(tidak ada proses di port {PORT})'")
    raise SystemExit("Gateway lokal belum benar; lihat log di atas. 'dengan token' = 502 berarti Ollama tidak menjawab.")

# 5) Tunnel Cloudflare ke gateway (bukan ke Ollama langsung)
step("Tunnel")
if not os.path.exists("/content/cloudflared"):
    sh("curl -sSLo /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/"
       "cloudflared-linux-amd64 && chmod +x /content/cloudflared")
tunnel_log = open("/content/tunnel.log", "w")
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", LOCAL], stdout=tunnel_log, stderr=tunnel_log,
                 start_new_session=True)
url = ""
for _ in range(60):
    found = [item for item in re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", tail("/content/tunnel.log", 200))
             if not item.startswith("https://api.")]  # api.trycloudflare.com muncul di pesan error
    if found:
        url = found[-1]
        break
    time.sleep(1)
if not url:
    print(tail("/content/tunnel.log", 30))
    raise SystemExit("Tunnel belum mendapat URL dalam 60 detik (trycloudflare kadang membatasi). Jalankan ulang sel "
                     "ini beberapa menit lagi; model tidak diunduh ulang.")

# 6) Cek lewat URL publik: hanya informasi (nama tunnel baru bisa belum ter-resolve dari dalam Colab)
step("Cek lewat URL publik (informasi)")
host = urllib.parse.urlsplit(url).hostname
public = {}
for _ in range(20):
    try:
        socket.getaddrinfo(host, 443)
    except socket.gaierror:
        time.sleep(3)
        continue
    public = checks(url)
    if public == EXPECTED:
        break
    time.sleep(3)
if not public:
    print("  Nama tunnel belum bisa di-resolve dari dalam Colab (normal untuk tunnel baru). Gateway lokal sudah")
    print("  lolos: tempel URL + token di dashboard, tunggu 1-2 menit, lalu klik 'Cek kesiapan' dari laptop.")
else:
    for name, code in public.items():
        print(f"  {name}: {code}")
    if public != EXPECTED:
        print("  Belum sesuai dari dalam Colab; tunggu 1-2 menit lalu klik 'Cek kesiapan' di dashboard.")

print()
print("=" * 60)
print("Tempel di dashboard SignalGate, tab Runtime & GPU -> GPU Colab:")
print("URL layanan  :", url)
print("Token gateway:", token)
print("=" * 60)
